# Step 1. 데이터 다운로드
아래 링크에서 korean-english-park.train.tar.gz 를 다운로드받아 한영 병렬 데이터를 확보합니다.

jungyeul/korean-parallel-corpora

# Step 0. 필수 라이브러리 및 디바이스 설정
가장 먼저 필요한 라이브러리를 가져오고 GPU(CUDA) 사용 여부를 확인합니다.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
import os
from konlpy.tag import Okt
from collections import Counter

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Step 1 & 2. 데이터 전처리 및 토큰화
중복을 제거하고 한글/영어 전처리와 Okt를 이용한 토큰화를 진행합니다. (데이터 파일 이름이 맞는지 꼭 확인해 주세요!)

In [3]:
path_to_ko = "work/korean-english/korean-english-park.train.ko"
path_to_en = "work/korean-english/korean-english-park.train.en"

# 1. 전처리 함수 정의
def preprocess_sentence(sentence, is_ko=False):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'\([^)]*\)', '', sentence)  # 괄호 내용 제거
    sentence = re.sub(r"[^a-zA-Z가-힣0-9?.!,]+", " ", sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = sentence.strip()
    
    if not is_ko:
        sentence = '<start> ' + sentence + ' <end>'
    return sentence

# 2. 커스텀 토크나이저 클래스
class Tokenizer:
    def __init__(self, num_words=12000):
        self.num_words = num_words
        self.word2index = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
        self.index2word = {0: "<pad>", 1: "<unk>", 2: "<start>", 3: "<end>"}
        self.word_count = Counter()
        
    def fit_on_texts(self, corpus):
        for tokens in corpus:
            self.word_count.update(tokens)
            
        most_common = self.word_count.most_common(self.num_words - 4)
        for word, _ in most_common:
            if word not in self.word2index:
                idx = len(self.word2index)
                self.word2index[word] = idx
                self.index2word[idx] = word
                
    def texts_to_sequences(self, corpus, max_len=40):
        sequences = []
        for tokens in corpus:
            seq = [self.word2index.get(token, self.word2index["<unk>"]) for token in tokens]
            if len(seq) < max_len:
                seq = seq + [self.word2index["<pad>"]] * (max_len - len(seq))
            else:
                seq = seq[:max_len]
            sequences.append(seq)
        return np.array(sequences)

# 3. 데이터 로드 및 정제 수행
print("데이터 로딩 중...")
with open(path_to_ko, "r", encoding="utf-8") as f: ko_data = f.read().splitlines()
with open(path_to_en, "r", encoding="utf-8") as f: en_data = f.read().splitlines()

cleaned_corpus = list(set(zip(ko_data, en_data)))
okt = Okt()
kor_corpus, eng_corpus = [], []

print("데이터 전처리 및 토큰화 진행 중 (시간이 다소 소요됩니다)...")
for ko, en in cleaned_corpus:
    ko_pre = preprocess_sentence(ko, is_ko=True)
    en_pre = preprocess_sentence(en, is_ko=False)
    
    ko_tokens = okt.morphs(ko_pre)
    en_tokens = en_pre.split()
    
    # 길이 40 이하만 필터링
    if len(ko_tokens) <= 40 and len(en_tokens) <= 40:
        kor_corpus.append(ko_tokens)
        eng_corpus.append(en_tokens)
        
print(f"필터링된 데이터 개수: {len(kor_corpus)}")

데이터 로딩 중...
데이터 전처리 및 토큰화 진행 중 (시간이 다소 소요됩니다)...
필터링된 데이터 개수: 66549


# Step 3. 텐서 변환 및 데이터로더 구축 (에러 수정 단계)
수정된 drop_last=True 인자를 반영하여 PyTorch용 데이터셋과 데이터로더를 만듭니다.

In [4]:
# 정수 인코딩
src_tokenizer = Tokenizer(num_words=12000)
src_tokenizer.fit_on_texts(kor_corpus)
src_tensor = src_tokenizer.texts_to_sequences(kor_corpus)

tgt_tokenizer = Tokenizer(num_words=12000)
tgt_tokenizer.fit_on_texts(eng_corpus)
tgt_tensor = tgt_tokenizer.texts_to_sequences(eng_corpus)

VOCAB_SIZE_KO = len(src_tokenizer.word2index)
VOCAB_SIZE_EN = len(tgt_tokenizer.word2index)

# 하이퍼파라미터
BATCH_SIZE = 64
EMBEDDING_DIM = 256
UNITS = 1024

# PyTorch 데이터셋 선언
class ParallelDataset(Dataset):
    def __init__(self, src_tensor, tgt_tensor):
        self.src_tensor = torch.tensor(src_tensor, dtype=torch.long)
        self.tgt_tensor = torch.tensor(tgt_tensor, dtype=torch.long)
    def __len__(self):
        return len(self.src_tensor)
    def __getitem__(self, idx):
        return self.src_tensor[idx], self.tgt_tensor[idx]

dataset = ParallelDataset(src_tensor, tgt_tensor)

# ⭐ drop_remainder 대신 drop_last=True 사용!
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
print("데이터로더 구축 완료!")

데이터로더 구축 완료!


# Step 4. 모델 설계 (Encoder, Attention, Decoder)
Bahdanau Attention 기반 신경망 네트워크를 빌드하고 그래픽카드(혹은 CPU)에 올립니다.

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, enc_units, batch_first=True)
        
    def forward(self, x):
        x = self.embedding(x)
        output, state = self.gru(x)
        return output, state

class BahdanauAttention(nn.Module):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(units, units)
        self.W2 = nn.Linear(units, units)
        self.V = nn.Linear(units, 1)
        
    def forward(self, query, values):
        query = query.transpose(0, 1)
        score = self.V(torch.tanh(self.W1(query) + self.W2(values)))
        attention_weights = torch.softmax(score, dim=1)
        context_vector = attention_weights * values
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim + dec_units, dec_units, batch_first=True)
        self.fc = nn.Linear(dec_units, vocab_size)
        self.attention = BahdanauAttention(dec_units)
        
    def forward(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        context_vector = context_vector.unsqueeze(1)
        x = torch.cat([context_vector, x], dim=-1)
        output, state = self.gru(x, hidden)
        output = output.squeeze(1)
        x = self.fc(output)
        return x, state, attention_weights

# 모델 인스턴스화
encoder = Encoder(VOCAB_SIZE_KO, EMBEDDING_DIM, UNITS).to(device)
decoder = Decoder(VOCAB_SIZE_EN, EMBEDDING_DIM, UNITS).to(device)

optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()))
criterion = nn.CrossEntropyLoss(ignore_index=0) # <pad> 토큰은 학습에서 제외

# Step 5. 훈련 및 번역 테스트 함수 생성
학습 중간 및 종료 후에 모델 성능을 직접 한글 예문으로 테스트할 수 있는 평가 로직과 훈련 루프입니다.

In [7]:
# 실시간 번역 결과 확인용 함수
def evaluate(sentence, encoder, decoder, src_tokenizer, tgt_tokenizer, okt, max_len=40):
    encoder.eval()
    decoder.eval()
    
    sentence = preprocess_sentence(sentence, is_ko=True)
    tokens = okt.morphs(sentence)
    
    seq = [src_tokenizer.word2index.get(token, src_tokenizer.word2index["<unk>"]) for token in tokens]
    if len(seq) < max_len:
        seq = seq + [src_tokenizer.word2index["<pad>"]] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
        
    inputs = torch.tensor([seq], dtype=torch.long).to(device)
    result = ''
    
    with torch.no_grad():
        enc_output, enc_hidden = encoder(inputs)
        dec_hidden = enc_hidden
        dec_input = torch.tensor([[tgt_tokenizer.word2index["<start>"]]], dtype=torch.long).to(device)
        
        for t in range(max_len):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            predicted_id = predictions.argmax(dim=-1).item()
            
            # 💡 index_word를 index2word로 수정했습니다!
            word = tgt_tokenizer.index2word.get(predicted_id, "<unk>")
            if word == "<end>":
                break
            result += word + ' '
            dec_input = torch.tensor([[predicted_id]], dtype=torch.long).to(device)
            
    return result.strip(), sentence

# 훈련 시작 (우선 테스트를 위해 5 Epoch로 설정)
EPOCHS = 5

print("학습을 시작합니다...")
for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    total_loss = 0
    
    for batch, (src, tgt) in enumerate(dataloader):
        src, tgt = src.to(device), tgt.to(device)
        loss = 0
        optimizer.zero_grad()
        
        enc_output, enc_hidden = encoder(src)
        dec_hidden = enc_hidden
        dec_input = tgt[:, 0].unsqueeze(1) # <start> 토큰 주입
        
        # Teacher Forcing
        for t in range(1, tgt.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += criterion(predictions, tgt[:, t])
            dec_input = tgt[:, t].unsqueeze(1)
            
        loss.backward()
        optimizer.step()
        
        total_loss += (loss.item() / tgt.shape[1])
        
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss / len(dataloader):.4f}')
    
    # 에폭마다 번역 품질 확인하기
    test_sentences = ["오렌지 주스는 맛있다.", "그는 학교에 갑니다.", "일곱 명의 사망자가 발생했다."]
    for s in test_sentences:
        res, src_text = evaluate(s, encoder, decoder, src_tokenizer, tgt_tokenizer, okt)
        print(f'  [테스트] 입력: {s} -> 번역 예측: {res}')

학습을 시작합니다...
Epoch 1/5 | Train Loss: nan
  [테스트] 입력: 오렌지 주스는 맛있다. -> 번역 예측: <unk> is not a <unk> .
  [테스트] 입력: 그는 학교에 갑니다. -> 번역 예측: he was not a <unk> .
  [테스트] 입력: 일곱 명의 사망자가 발생했다. -> 번역 예측: the death toll were heard .
Epoch 2/5 | Train Loss: nan
  [테스트] 입력: 오렌지 주스는 맛있다. -> 번역 예측: the <unk> is a tricky .
  [테스트] 입력: 그는 학교에 갑니다. -> 번역 예측: he was <unk> .
  [테스트] 입력: 일곱 명의 사망자가 발생했다. -> 번역 예측: the death toll was a toll of a storm .
Epoch 3/5 | Train Loss: nan
  [테스트] 입력: 오렌지 주스는 맛있다. -> 번역 예측: <unk> is not a <unk> pig .
  [테스트] 입력: 그는 학교에 갑니다. -> 번역 예측: he <unk> .
  [테스트] 입력: 일곱 명의 사망자가 발생했다. -> 번역 예측: the death toll at norris hall .
Epoch 4/5 | Train Loss: nan
  [테스트] 입력: 오렌지 주스는 맛있다. -> 번역 예측: <unk> is not a <unk> .
  [테스트] 입력: 그는 학교에 갑니다. -> 번역 예측: he <unk> .
  [테스트] 입력: 일곱 명의 사망자가 발생했다. -> 번역 예측: the death toll was a factor .
Epoch 5/5 | Train Loss: nan
  [테스트] 입력: 오렌지 주스는 맛있다. -> 번역 예측: <unk> has not been <unk> .
  [테스트] 입력: 그는 학교에 갑니다. -> 번역 예측: he was a <unk> .
  [테스트] 입력: 일곱 명의 